In [9]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()

X = digits.images / 16.0
y = digits.target

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (1257, 8, 8)
Validation: (270, 8, 8)
Testing: (270, 8, 8)


In [10]:
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=64, shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val), batch_size=64
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test), batch_size=64
)

In [11]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.fc(self.conv(x))


cnn = CNNModel()

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)

for epoch in range(10):
    cnn.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = cnn(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/10, Loss: {loss.item():.4f}")

Epoch 1/10, Loss: 2.2374
Epoch 2/10, Loss: 2.0331
Epoch 3/10, Loss: 1.6462
Epoch 4/10, Loss: 1.0469
Epoch 5/10, Loss: 0.7684
Epoch 6/10, Loss: 0.3746
Epoch 7/10, Loss: 0.6070
Epoch 8/10, Loss: 0.4005
Epoch 9/10, Loss: 0.2602
Epoch 10/10, Loss: 0.3043


In [13]:
from sklearn.metrics import accuracy_score

cnn.eval()
predictions = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = cnn(images)
        predictions.extend(outputs.argmax(1).numpy())

print("Validation Accuracy:", accuracy_score(y_val.numpy(), predictions))

Validation Accuracy: 0.9


In [14]:
class ViTModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Linear(4, 32)
        self.position = nn.Parameter(torch.randn(1, 16, 32))

        layer = nn.TransformerEncoderLayer(
            d_model=32, nhead=4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)

        self.fc = nn.Linear(32, 10)

    def forward(self, x):
        x = x.unfold(2, 2, 2).unfold(3, 2, 2)
        x = x.contiguous().view(x.size(0), 16, 4)
        x = self.embedding(x) + self.position
        x = self.transformer(x)
        return self.fc(x.mean(dim=1))


vit = ViTModel()

In [15]:
vit_criterion = nn.CrossEntropyLoss()
vit_optimizer = torch.optim.Adam(vit.parameters(), lr=0.001)

for epoch in range(10):
    vit.train()

    for images, labels in train_loader:
        vit_optimizer.zero_grad()
        outputs = vit(images)
        loss = vit_criterion(outputs, labels)
        loss.backward()
        vit_optimizer.step()

    print(f"Epoch {epoch+1}/10, Loss: {loss.item():.4f}")

Epoch 1/10, Loss: 2.3347
Epoch 2/10, Loss: 2.2726
Epoch 3/10, Loss: 2.2566
Epoch 4/10, Loss: 2.1797
Epoch 5/10, Loss: 1.8255
Epoch 6/10, Loss: 1.5791
Epoch 7/10, Loss: 1.3212
Epoch 8/10, Loss: 1.1920
Epoch 9/10, Loss: 0.7467
Epoch 10/10, Loss: 0.8365


In [16]:
vit.eval()
vit_predictions = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = vit(images)
        vit_predictions.extend(outputs.argmax(1).numpy())

print("ViT Validation Accuracy:",
      accuracy_score(y_val.numpy(), vit_predictions))

ViT Validation Accuracy: 0.8296296296296296


In [17]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

cnn.eval()
vit.eval()

cnn_preds = []
vit_preds = []
actual = []

with torch.no_grad():
    for images, labels in test_loader:
        cnn_preds.extend(cnn(images).argmax(1).numpy())
        vit_preds.extend(vit(images).argmax(1).numpy())
        actual.extend(labels.numpy())

print("CNN Test Accuracy:", accuracy_score(actual, cnn_preds))
print("CNN Macro F1:", f1_score(actual, cnn_preds, average="macro"))

print("ViT Test Accuracy:", accuracy_score(actual, vit_preds))
print("ViT Macro F1:", f1_score(actual, vit_preds, average="macro"))

CNN Test Accuracy: 0.9185185185185185
CNN Macro F1: 0.9170642339461447
ViT Test Accuracy: 0.8407407407407408
ViT Macro F1: 0.8326181195841448


In [18]:
print("CNN Confusion Matrix:")
print(confusion_matrix(actual, cnn_preds))

print("\nViT Confusion Matrix:")
print(confusion_matrix(actual, vit_preds))

CNN Confusion Matrix:
[[27  0  0  0  0  0  0  0  0  0]
 [ 0 22  2  1  0  0  0  0  2  1]
 [ 0  0 24  0  0  0  0  0  2  0]
 [ 0  0  0 28  0  0  0  0  0  0]
 [ 0  0  0  0 26  0  1  0  0  0]
 [ 0  0  0  0  0 27  0  0  0  0]
 [ 0  0  0  0  0  1 26  0  0  0]
 [ 0  0  0  0  1  0  0 26  0  0]
 [ 0  6  0  0  0  0  0  1 19  0]
 [ 0  0  0  0  2  0  0  2  0 23]]

ViT Confusion Matrix:
[[25  0  0  0  1  1  0  0  0  0]
 [ 0 22  0  0  0  0  0  0  3  3]
 [ 0  0 23  0  0  0  0  2  1  0]
 [ 0  0  0 27  0  0  0  0  0  1]
 [ 0  0  0  0 23  0  1  2  1  0]
 [ 0  0  0  0  0 27  0  0  0  0]
 [ 0  0  0  0  1  0 26  0  0  0]
 [ 0  1  0  0  0  0  0 26  0  0]
 [ 1  7  0  0  0  3  1  3 10  1]
 [ 0  0  0  1  0  7  0  1  0 18]]


In [19]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)

print("Baseline Accuracy:",
      accuracy_score(y_test.numpy(), baseline_preds))

Baseline Accuracy: 0.1037037037037037


In [20]:
torch.save(cnn.state_dict(), "cnn_model.pth")
torch.save(vit.state_dict(), "vit_model.pth")

print("Both models saved successfully!")

Both models saved successfully!


In [21]:
import json

results = {
    "CNN Test Accuracy": accuracy_score(actual, cnn_preds),
    "CNN Macro F1": f1_score(actual, cnn_preds, average="macro"),
    "ViT Test Accuracy": accuracy_score(actual, vit_preds),
    "ViT Macro F1": f1_score(actual, vit_preds, average="macro"),
    "Baseline Accuracy": accuracy_score(y_test.numpy(), baseline_preds)
}

with open("evaluation_results.json", "w") as file:
    json.dump(results, file, indent=4)

print("Evaluation results saved!")

Evaluation results saved!


In [23]:
import json

results = {
    "CNN Test Accuracy": 0.9185,
    "CNN Macro F1": 0.9171,
    "ViT Test Accuracy": 0.8407,
    "ViT Macro F1": 0.8326,
    "Majority Baseline Accuracy": 0.1037
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Evaluation results saved!")

Evaluation results saved!
